In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
print("Loading imports...")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
# parse_dates ensures date columns are loaded as datetime objects, saving a conversion step later
train = pd.read_csv('store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
stores = pd.read_csv('store-sales-time-series-forecasting/stores.csv')
holidays_events = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
oil = pd.read_csv('store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])


# =========================================================
# 3. DATA MERGING & CHECK FOR MISSING VALUES
# =========================================================
print("Merging data...")
# Merge store metadata (city, state, type, cluster) into our main sets
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

# Prepare Holidays
# In Ecuador, if a holiday falls on a weekend, it is often 'transferred' to a weekday.
# We filter out the original date of transferred holidays so we don't count them twice.
valid_holidays = holidays_events[holidays_events['transferred'] == False].drop_duplicates(subset=['date'])
valid_holidays = valid_holidays[['date', 'type']].rename(columns={'type': 'holiday_type'})

train = train.merge(valid_holidays, on='date', how='left')
test = test.merge(valid_holidays, on='date', how='left')

# Combine train and test to ensure consistent feature engineering
# This prevents errors where a specific category (e.g., a specific holiday) appears 
# in the test set but not the training set, which would break the OneHotEncoder.
train['is_test'] = False
test['is_test'] = True
test['sales'] = 0.0 # Placeholder target for the test set

df = pd.concat([train, test], ignore_index=True)

# The competition metric is Root Mean Squared Logarithmic Error (RMSLE).
# By applying log1p (log(1 + x)) to the target now, we can just use standard 
# RMSE as our loss function in the model.
df['log1p_sales'] = np.log1p(df['sales'])

print("Checking missing values...")
print(df.isnull().sum())
print(oil.isnull().sum())

# Examining oil data closer
# Getting all the unique dates
all_dates = df['date'].unique()
# Creating a dataframe with all the unique dates
all_dates_df = pd.DataFrame({'date': all_dates})
# Creating a dataframe for the oil data which has all the dates
oil_all_dates = all_dates_df.merge(oil, on='date', how='left')

print(oil_all_dates.isnull().sum())

print(oil_all_dates.shape)

print(f'{100*(525/1700):.2f}% of the oil dates are missing')



In [ ]:
# =========================================================
# 4. PLOTTING THE LOG1P_SALES DATA
# =========================================================
# Calculate the average log1p(sales) by date and family
# This averages the sales across all stores for each family on each day
daily_family_avg = df[~df['is_test']].groupby(['date', 'family'])['log1p_sales'].mean().reset_index()

# Generate the plots
# Using seaborn's relplot to create a grid of separate plots for each family
g = sns.relplot(
    data=daily_family_avg,
    x='date', 
    y='log1p_sales',
    col='family',       # Create a different subplot for each family
    col_wrap=5,         # Number of plots per row
    kind='line', 
    height=3.5,         # Height of each individual subplot
    aspect=1.5,         # Width-to-height ratio of each subplot
    facet_kws={'sharey': False} # Set to True if you want them all on the same Y-axis scale
)

# Formatting the plots
g.set_axis_labels("Date", "Avg log1p(Sales)")
g.set_titles(col_template="{col_name}", size=12)

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 4. PLOTTING THE OIL DATA
# =========================================================
oil_all_dates.plot(kind='line', x='date', y='dcoilwtico')
plt.show()


In [ ]:
# =========================================================
# 5. EXAMINING STORE_NBR, CLUSTERS, AND TYPE
# =========================================================
print('\nPrinting store numbers...')
store_numbers = df['store_nbr'].unique().tolist()
store_numbers.sort()
print(store_numbers)

print('\nPrinting store clusters...')
store_cluster = df['cluster'].unique().tolist()
store_cluster.sort()
print(store_cluster)

print('\nPrinting store types...')
store_type = df['type'].unique().tolist()
store_type.sort()
print(store_type)

print('\nPrinting store clusters with multiple store types...')
for i in range(1,18):
    temp = train[train['cluster']==i]['type'].unique().tolist()
    if len(temp)==0:
        print(f'cluster {i} contains no stores of any type')
    elif len(temp)==1:
        pass
    else:
        print(f'cluster {i} contains stores of type {temp}')